## 1. Initialize Project Environment
Import libraries for phylogenetic tree construction and analysis.

> **Note**: This notebook uses the **MSA-aligned sequences** from Task 1 to build a biologically correct NJ tree.

In [1]:
from __future__ import annotations

import json
import logging
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List

import pandas as pd
from Bio import Phylo, AlignIO
from Bio.Phylo.TreeConstruction import DistanceCalculator, DistanceTreeConstructor
from Bio.Align import MultipleSeqAlignment

logging.basicConfig(level=logging.INFO, format="[%(levelname)s] %(message)s")

## 2. Define Configuration Parameters
Centralize paths and tree construction options.

In [2]:
@dataclass
class TreeConfig:
    handle: str
    msa_path: Path = Path("artifacts/task1_msa_result.clustal")  # Use MSA from Task 1
    export_dir: Path = Path("artifacts")
    distance_model: str = "identity"

    def describe(self) -> Dict[str, str]:
        info = asdict(self)
        info["msa_path"] = str(info["msa_path"])
        info["export_dir"] = str(info["export_dir"])
        return info


CONFIG = TreeConfig(handle="AndreiCod")
CONFIG.describe()

{'handle': 'AndreiCod',
 'msa_path': 'artifacts/task1_msa_result.clustal',
 'export_dir': 'artifacts',
 'distance_model': 'identity'}

In [3]:
def load_alignment(cfg: TreeConfig) -> MultipleSeqAlignment:
    """Load MSA from Task 1."""
    if not cfg.msa_path.exists():
        raise FileNotFoundError(f"MSA not found: {cfg.msa_path}. Run Task1 first.")

    alignment = AlignIO.read(cfg.msa_path, "clustal")
    logging.info(
        "Loaded MSA: %d sequences × %d positions",
        len(alignment),
        alignment.get_alignment_length(),
    )
    return alignment


alignment = load_alignment(CONFIG)
print(
    f"Loaded alignment: {len(alignment)} sequences × {alignment.get_alignment_length()} bp"
)
print(f"\nSequences in alignment:")
for rec in alignment:
    print(f"  {rec.id}")

[INFO] Loaded MSA: 10 sequences × 1859 positions


Loaded alignment: 10 sequences × 1859 bp

Sequences in alignment:
  NM_000546.6
  NM_011640.3
  NM_131327.2
  NM_001003210.1
  NM_213824.3
  NM_174201.2
  NM_205264.1
  NM_030989.3
  XM_016931470.3
  NM_001047151.2


## 3. Build Neighbor-Joining Tree
Use Biopython's DistanceCalculator on the **properly aligned** sequences.

In [4]:
def build_nj_tree(alignment: MultipleSeqAlignment, model: str = "identity"):
    """Build Neighbor-Joining tree from MSA using Biopython."""
    calculator = DistanceCalculator(model)
    distance_matrix = calculator.get_distance(alignment)

    constructor = DistanceTreeConstructor()
    tree = constructor.nj(distance_matrix)

    return tree, distance_matrix


tree, distance_matrix = build_nj_tree(alignment, CONFIG.distance_model)

print("Distance Matrix (from proper MSA):")
print(distance_matrix)

Distance Matrix (from proper MSA):
NM_000546.6 0.000000
NM_011640.3 0.281872    0.000000
NM_131327.2 0.543303    0.564282    0.000000
NM_001003210.1  0.285637    0.339968    0.495966    0.000000
NM_213824.3 0.203335    0.300699    0.538999    0.284562    0.000000
NM_174201.2 0.193115    0.309844    0.553523    0.295320    0.150619    0.000000
NM_205264.1 0.561054    0.583109    0.554061    0.529317    0.556213    0.556213    0.000000
NM_030989.3 0.263583    0.134481    0.533082    0.337816    0.288327    0.297472    0.566971    0.000000
XM_016931470.3  0.129640    0.346961    0.574502    0.284024    0.308768    0.291017    0.577730    0.342657    0.000000
NM_001047151.2  0.105971    0.322754    0.518020    0.295858    0.230231    0.238300    0.538462    0.287251    0.226466    0.000000
    NM_000546.6 NM_011640.3 NM_131327.2 NM_001003210.1  NM_213824.3 NM_174201.2 NM_205264.1 NM_030989.3 XM_016931470.3  NM_001047151.2


In [5]:
# Species name mapping for better interpretation
SPECIES_NAMES = {
    "NM_000546.6": "Human (H. sapiens)",
    "NM_011640.3": "Mouse (M. musculus)",
    "NM_131327.2": "Zebrafish (D. rerio)",
    "NM_001003210.1": "Dog (C. familiaris)",
    "NM_213824.3": "Pig (S. scrofa)",
    "NM_174201.2": "Cattle (B. taurus)",
    "NM_205264.1": "Chicken (G. gallus)",
    "NM_030989.3": "Rat (R. norvegicus)",
    "XM_016931470.3": "Chimpanzee (P. troglodytes)",
    "NM_001047151.2": "Rhesus (M. mulatta)",
}

# Create labeled tree for display
import copy

tree_labeled = copy.deepcopy(tree)
for clade in tree_labeled.find_clades():
    if clade.name and clade.name in SPECIES_NAMES:
        clade.name = SPECIES_NAMES[clade.name]

print("=== Neighbor-Joining Tree ===\n")
Phylo.draw_ascii(tree_labeled)

=== Neighbor-Joining Tree ===

      ________ Cattle (B. taurus)
  ___|
 |   |________ Pig (S. scrofa)
 |
 |    ______________ Dog (C. familiaris)
 | __|
 ||  |            _______________________________ Chicken (G. gallus)
 ||  |___________|
_||              |____________________________ Zebrafish (D. rerio)
 ||
 ||           ______ Rat (R. norvegicus)
 ||__________|
 |           |________ Mouse (M. musculus)
 |
 |    _______ Rhesus (M. mulatta)
 |___|
     |  __________ Chimpanzee (P. troglodytes)
     |_|
       |___ Human (H. sapiens)



## 4. Analyze Tree Clusters
Extract cluster statistics for notes.md (insights documented there, not in code).

In [6]:
def extract_tree_stats(tree) -> Dict:
    """Extract quantitative tree statistics."""
    terminals = list(tree.get_terminals())
    internals = list(tree.get_nonterminals())

    # Get branch lengths
    branch_lengths = [c.branch_length for c in tree.find_clades() if c.branch_length]

    stats = {
        "num_terminals": len(terminals),
        "num_internal_nodes": len(internals),
        "total_tree_length": round(sum(branch_lengths), 4) if branch_lengths else None,
        "mean_branch_length": round(sum(branch_lengths) / len(branch_lengths), 4)
        if branch_lengths
        else None,
        "min_branch_length": round(min(branch_lengths), 4) if branch_lengths else None,
        "max_branch_length": round(max(branch_lengths), 4) if branch_lengths else None,
        "terminal_names": [t.name for t in terminals],
    }

    return stats


tree_stats = extract_tree_stats(tree)

print("Tree Statistics:")
for k, v in tree_stats.items():
    if k != "terminal_names":
        print(f"  {k}: {v}")
print(f"\nTerminals: {len(tree_stats['terminal_names'])} species")

Tree Statistics:
  num_terminals: 10
  num_internal_nodes: 8
  total_tree_length: 1.5311
  mean_branch_length: 0.0901
  min_branch_length: 0.0152
  max_branch_length: 0.2862

Terminals: 10 species


## 5. Export Results
Save tree in Newick format and interpretation to artifacts.

In [7]:
EXPORT_DIR = CONFIG.export_dir
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# Save tree in Newick format (with original accession IDs)
tree_path = EXPORT_DIR / "task2_nj_tree.nwk"
Phylo.write(tree, tree_path, "newick")
print(f"[OK] NJ tree saved to: {tree_path.resolve()}")

# Save tree stats as JSON
stats_path = EXPORT_DIR / "task2_tree_stats.json"
with open(stats_path, "w") as f:
    json.dump(tree_stats, f, indent=2)
print(f"[OK] Tree statistics saved to: {stats_path.resolve()}")

print(f"\nArtifacts saved to {EXPORT_DIR.resolve()}")

[OK] NJ tree saved to: /home/rbals/git/daha-bdhb/BDHB-lab/labs/04_phylogenetics/assignments/artifacts/task2_nj_tree.nwk
[OK] Tree statistics saved to: /home/rbals/git/daha-bdhb/BDHB-lab/labs/04_phylogenetics/assignments/artifacts/task2_tree_stats.json

Artifacts saved to /home/rbals/git/daha-bdhb/BDHB-lab/labs/04_phylogenetics/assignments/artifacts
